# Notebook 03: Mask Fine-Tuning with Classification Heads

In this notebook, we perform end-to-end training but **freeze the entire Moirai encoder except for the mask tokens**. 

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display

from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [16]#[8, 16, 32, 64]

lr_grid=[5e-5]
wd_grid=[0.05]

BATCH_SIZE = 32

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

## 1. Baseline: Linear Model on Mask Embedding Only

In [ ]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    print(patch)
    _, test_acc = universal_grid_search(
        model_class=MaskOnlyFinetunerWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid= wd_grid, lr_grid = lr_grid,
    )
    df_mask_only.loc["Mask Only", patch] = test_acc

16
LR=5e-05 | WD=0.05
Val Loss: 1.3166
Acc on Test Set : 0.5702



Patch Size,16
Mask Only,0.5702


In [ ]:
display(df_mask_only.astype(float).round(4))

## 2. Baseline: Mean Pooling on Full Sequence (Context + Mask)
We average all patches (including the unfrozen mask) before passing them to a linear classifier.

In [4]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = universal_grid_search(
        model_class=HeadFinetunerWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid= wd_grid, lr_grid = lr_grid
    )
    df_mean_pool.loc["Mean Pooling", patch] = test_acc

LR=5e-05 | WD=0.05


Val Loss: 1.2490
Acc on Test Set : 0.5856



In [ ]:
display(df_mean_pool.astype(float).round(4))

## 3. Advanced Pooling: Attention & MHA (Context + Mask)
We use Attention mechanisms to dynamically weight the patches. The network can now learn to pay attention to specific context patches AND the fine-tuned mask patch.

In [5]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, acc_attn = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid= wd_grid, lr_grid = lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_attn

        # 2. Multi-Head Attention
        _, acc_mha = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid= wd_grid, lr_grid = lr_grid
        )
        df_adv_single.loc[(model_names_single[1], mode), patch] = acc_mha

LR=5e-05 | WD=0.05
Val Loss: 1.3166
Acc on Test Set : 0.5629

LR=5e-05 | WD=0.05
Val Loss: 1.1414
Acc on Test Set : 0.6180

LR=5e-05 | WD=0.05
Val Loss: 1.3055
Acc on Test Set : 0.5665

LR=5e-05 | WD=0.05


Val Loss: 1.1439
Acc on Test Set : 0.6139

LR=5e-05 | WD=0.05
Val Loss: 1.2459
Acc on Test Set : 0.5803

LR=5e-05 | WD=0.05
Val Loss: 1.1178
Acc on Test Set : 0.6245

LR=5e-05 | WD=0.05
Val Loss: 1.2456
Acc on Test Set : 0.5799

LR=5e-05 | WD=0.05
Val Loss: 1.1157
Acc on Test Set : 0.6225



In [6]:
df_adv_single.astype(float).round(4)

Patch Size                                8       16
Model            Mode                               
Attention (Base) shared_context       0.5629  0.5803
                 independent_context  0.5665  0.5799
MHA (Heads=16)   shared_context       0.6180  0.6245
                 independent_context  0.6139  0.6225

## 4. Ablation Study: Number of Attention Heads
Testing the impact of `num_heads` on the Single-Scale MHA model (with mask fine-tuning) for a fixed patch size of 16.

In [8]:
PATCH = 16
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [16, 32, 64, 128] 

df_heads_ablation = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation.index.name = "Mode"
df_heads_ablation.columns.name = f"Num Heads (Patch {PATCH})"

for mode in MODES:
    for heads in HEADS_TO_TEST:
        
        _, test_acc = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {
                    "num_vars": NUM_VARS, 
                    "num_classes": num_classes, 
                    "patch_mode": mode, 
                    "num_heads": heads
                },
                "patch_size": PATCH, 
                "num_vars": NUM_VARS, 
                "size": SIZE, 
                "remove_last_patch": False
            },
            train_loader=tr_loader, 
            val_loader=va_loader, 
            test_loader=te_loader, 
            device=DEVICE,
            wd_grid = wd_grid, lr_grid=lr_grid
        )
        
        df_heads_ablation.loc[mode, heads] = test_acc

LR=5e-05 | WD=0.05
Val Loss: 1.1127
Acc on Test Set : 0.6139

LR=5e-05 | WD=0.05
Val Loss: 1.1166
Acc on Test Set : 0.6208

LR=5e-05 | WD=0.05
Val Loss: 1.0961
Acc on Test Set : 0.6148

LR=5e-05 | WD=0.05
Val Loss: 1.1028
Acc on Test Set : 0.6253

LR=5e-05 | WD=0.05
Val Loss: 1.1034
Acc on Test Set : 0.6277

LR=5e-05 | WD=0.05
Val Loss: 1.1045
Acc on Test Set : 0.6249

LR=5e-05 | WD=0.05
Val Loss: 1.0873
Acc on Test Set : 0.6277

LR=5e-05 | WD=0.05
Val Loss: 1.0990
Acc on Test Set : 0.6257



Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.6139,0.6208,0.6148,0.6253
independent_context,0.6277,0.6249,0.6277,0.6257


In [9]:
display(df_heads_ablation.astype(float).round(4))

Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.6139,0.6208,0.6148,0.6253
independent_context,0.6277,0.6249,0.6277,0.6257


# 5. Hierarchical

In [10]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_hier = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid = wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = acc_hier

LR=5e-05 | WD=0.05
Val Loss: 1.2776
Acc on Test Set : 0.5775

LR=5e-05 | WD=0.05
Val Loss: 1.2180
Acc on Test Set : 0.6050

LR=5e-05 | WD=0.05
Val Loss: 1.2401
Acc on Test Set : 0.5860

LR=5e-05 | WD=0.05
Val Loss: 1.2129
Acc on Test Set : 0.5994



In [11]:
display(df_hierarchical.astype(float).round(4))

Patch Size,8,16
Mode,,
shared_context,0.5775,0.5860
independent_context,0.6050,0.5994
